# 04 · Plotting with gog4j

Numbers became a table; now the table becomes a picture. **gog4j** is ggplot2
for JavaFX. The grammar is three pieces:

* **data** — here a dflib `DataFrame`, read straight from the Parquet
* **aes** — which columns drive x, y, colour, …
* **geoms** — how to draw them (points, lines, bars, density, …)

Build a plot with `Ggplot.ggplot(...)`, add layers with `.geoms(...)`, then
`display(...)` it.

> **Workaround: every plot needs an explicit height.** gog4j sets no default
> height on a `Plot`, so a displayed plot would otherwise collapse to a sliver.
> Each cell here sets `plot.setPrefHeight(400)` before `display(...)`.

In [0]:
addDependency("org.dflib:dflib:2.0.0-M7");
addDependency("org.dflib:dflib-parquet:2.0.0-M7");
addDependency("org.jtaccuino:gog4j:0.5.0");
addDependency("org.jtaccuino:gog4j-dflib:0.5.0");

In [0]:
// the dflib extension gives us println(DataFrame) - a real table
use("dflib");

In [0]:
import org.dflib.DataFrame;
import org.dflib.parquet.Parquet;
import static org.dflib.Exp.*;
import org.jtaccuino.gog.*;
import org.jtaccuino.gog.coord.Coord2D;
import org.jtaccuino.gog.labs.Labs;
import org.jtaccuino.gog.theme.Theme;
import java.nio.file.Files;
import java.nio.file.Path;

// The notebook may live in a sub-folder (exercises/, bonus/, ...), so walk up
// from the notebook's own folder to find the repository's data/ directory.
Path dataset() {
    for (Path p = cwd.toAbsolutePath(); p != null; p = p.getParent()) {
        Path candidate = p.resolve("data/celestrak_gp_catalog.parquet");
        if (Files.isRegularFile(candidate)) return candidate;
    }
    throw new IllegalStateException("data/celestrak_gp_catalog.parquet not found above " + cwd);
}

var full = Parquet.loader().load(dataset());

// Project to just the fields this module plots - reading and keeping only the
// columns you need is the columnar habit (and keeps the frame small).
var table = full.cols("inclination", "eccentricity", "period_min",
        "apogee_km", "orbit_class", "satcat_object_type").select();

## 1 · Data + aes + geoms

One point per satellite: inclination across, apogee up, coloured by orbit class.
That single `color(...)` is the whole point of a grammar of graphics.

In [0]:
Plot<DataFrame> p = Ggplot.ggplot(table,
        Aes.aes().x("inclination").y("apogee_km").color("orbit_class"))
    .geoms(Geoms.point());

In [0]:
// ── given ── look at it
p.setPrefHeight(400);
display(p);

## 2 · Labels

A plot without axis labels is a riddle. `Labs.labs(...)` answers it.

In [0]:
p = p.labs(Labs.labs("Everything in orbit", "Inclination (deg)", "Apogee (km)"));

In [0]:
p.setPrefHeight(400);
display(p);

## 3 · Theme

A theme changes every colour at once. On a projector, dark wins.

In [0]:
p = p.theme(Theme.theme_dark());

In [0]:
p.setPrefHeight(400);
display(p);

## 4 · Facets

Faceting splits one plot into a small multiple per category — the fastest way to
see that the classes are genuinely different populations.

In [0]:
p = p.facets(Facets.wrap("orbit_class", 2));

In [0]:
p.setPrefHeight(400);
display(p);

## 5 · Reference lines

The geostationary belt sits at 35 786 km. `Geoms.hline(...)` draws it, and one
glance then explains the horizontal stripe of GEO objects.

In [0]:
p = p.geoms(Geoms.point(), Geoms.hline(35786));

In [0]:
p.setPrefHeight(400);
display(p);

## The bridge to machine learning

Everything so far *described* the catalog. Now predict something from it. Can the
orbital elements alone tell debris from a working satellite?

`satcat_object_type` gives the ground truth: `PAY` (payload) or `DEB` / `R/B`
(debris). The features are the elements — inclination, eccentricity, period, mean
motion, drag. We stop at the train/test split — a labelled feature matrix, ready
for a model.

In [0]:
// ── given ── turn it into a supervised problem
var labelled = full.rows(
        $str("satcat_object_type").eq("PAY")
            .or($str("satcat_object_type").eq("DEB"))
            .or($str("satcat_object_type").eq("R/B")))
    .select();

println("labelled %s of %s rows", String.format(java.util.Locale.ROOT, "%,d", labelled.height()),
        String.format(java.util.Locale.ROOT, "%,d", full.height()));
println(labelled.group("satcat_object_type").agg($col("satcat_object_type"), count()));

In [0]:
// ── given ── features and a deterministic split
var features = labelled
    .cols("inclination", "eccentricity", "period_min", "mean_motion", "bstar")
    .select();

int split = (int) (features.height() * 0.8);
var train = features.rowsRange(0, split).select();
var test  = features.rowsRange(split, features.height()).select();

println("train %s   test %s   features %d",
        String.format(java.util.Locale.ROOT, "%,d", train.height()),
        String.format(java.util.Locale.ROOT, "%,d", test.height()),
        features.width());
println("→ a feature matrix and a labelled split, ready for a model.");

## ★ BONUS 1 — Pairs and composition

Two dozen features at once, as a scatterplot matrix. `matrixPlot` does the grid;
`composedPlot` arranges finished figures side by side.

In [0]:
// ── ★ BONUS 1 ─────────────────────────────────────────────────────────
var pairs = Ggplot.matrixPlot(table,
    Aes.aes().color("orbit_class"),
    "inclination", "eccentricity", "period_min", "apogee_km");
pairs.setPrefHeight(400);
display(pairs);

## ★ BONUS 2 — A three-dimensional orbit cloud

Add inclination, period and eccentricity as x/y/z and let the points fill the
volume.

In [0]:
// ── ★ BONUS 2 ─────────────────────────────────────────────────────────
var cloud = Ggplot.ggplot3d(table,
        Aes.aes().x("inclination").y("period_min").z("eccentricity").color("orbit_class"))
    .geoms(Geoms.point3d());
cloud.setPrefHeight(400);
display(cloud);

## ★ BONUS 3 — The wall chart

A filled 2-D density instead of 21 000 points, with the GEO belt drawn on top.
This is the figure that goes on the poster.

In [0]:
// ── ★ BONUS 3 ─────────────────────────────────────────────────────────
// Apogee spans four orders of magnitude and the geostationary belt is a thin
// ring, so zoom both axes onto it and tighten the bandwidth so the density
// does not smear across the whole inclination range.
var wall = Ggplot.ggplot(table,
        Aes.aes().x("inclination").y("apogee_km"))
    .coord(Coord2D.cartesian().xlim(0, 10).ylim(10000, 80000))
    .geoms(Geoms.density2dFilled().adjust(0.5), Geoms.hline(35786))
    .labs(Labs.labs("Where satellites live", "Inclination (deg)", "Apogee (km)"))
    .theme(Theme.theme_dark());
wall.setPrefHeight(400);
display(wall);

That is the whole pipeline: **dflib** read and shaped it, **gog4j** drew it, and
you left a labelled train/test split on the table for the model.

Back to the [overview](../narrative/00-overview.md).